# Module 2: Advanced Fitness Function Design

## Learning Objectives

By completing this notebook, you will:
1. Design fitness functions that balance multiple objectives
2. Implement parsimony pressure to prevent overfitting
3. Create domain-specific fitness for time series forecasting
4. Handle constraints through penalty functions
5. Implement fitness approximation for computational efficiency
6. Use Pareto optimization for true multi-objective problems

## Prerequisites

- Module 1 completed (GA fundamentals)
- Understanding of overfitting and cross-validation
- Basic time series concepts

## Estimated Time: 90 minutes

---

In [ ]:
learning_objectives(["Design fitness functions that balance multiple objectives", "Implement parsimony pressure to prevent overfitting", "Create domain-specific fitness for time series forecasting", "Handle constraints through penalty functions", "Implement fitness approximation for computational efficiency", "Use Pareto optimization for true multi-objective problems"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes, make_regression
from sklearn.model_selection import cross_val_score, TimeSeriesSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from typing import List, Tuple, Callable, Dict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

In [ ]:
apply_course_theme()
apply_plot_theme()

### Setup: Individual Class and Basic Functions

In [ ]:
# Purpose: Define Individual class for feature selection
# Key Concept: Binary chromosome representation

class Individual:
    """Represents a candidate solution for feature selection."""
    
    def __init__(self, chromosome: np.ndarray):
        self.chromosome = chromosome.astype(int)
        self.fitness = None
        self.objectives = {}  # Store multiple objectives
    
    @classmethod
    def random(cls, n_features: int):
        chromosome = np.random.randint(0, 2, size=n_features)
        return cls(chromosome)
    
    def selected_features(self) -> np.ndarray:
        return np.where(self.chromosome == 1)[0]
    
    def n_selected(self) -> int:
        return np.sum(self.chromosome)
    
    def copy(self):
        new_ind = Individual(self.chromosome.copy())
        new_ind.fitness = self.fitness
        new_ind.objectives = self.objectives.copy()
        return new_ind
    
    def __repr__(self):
        return f"Individual(features={self.n_selected()}, fitness={self.fitness:.4f if self.fitness else 'None'})"

print("Individual class defined!")

In [ ]:
section_divider("Multi-Objective Fitness Functions")

## 1. Multi-Objective Fitness Functions

### Key Concept: Real-world problems often have multiple conflicting objectives.

For feature selection:
1. **Maximize accuracy**: Predictive performance
2. **Minimize features**: Interpretability and efficiency
3. **Minimize redundancy**: Feature correlation
4. **Minimize cost**: Data collection expense (domain-specific)

In [ ]:
callout("Real-world problems often have multiple conflicting objectives.", kind="insight")

### 1.1 Weighted Sum Approach

Combine objectives using weighted sum: `fitness = w1*obj1 + w2*obj2 + ...`

In [ ]:
# Purpose: Implement weighted sum multi-objective fitness
# Key Concept: Weights express relative importance of objectives

def weighted_sum_fitness(individual: Individual, 
                        X: pd.DataFrame, 
                        y: np.ndarray,
                        w_accuracy: float = 1.0,
                        w_parsimony: float = 0.3,
                        w_redundancy: float = 0.2) -> float:
    """
    Multi-objective fitness using weighted sum.
    
    Parameters
    ----------
    individual : Individual
        Candidate solution
    X : DataFrame
        Feature matrix
    y : array
        Target variable
    w_accuracy : float
        Weight for accuracy (default 1.0)
    w_parsimony : float
        Weight for parsimony penalty (default 0.3)
    w_redundancy : float
        Weight for redundancy penalty (default 0.2)
    
    Returns
    -------
    fitness : float
        Combined fitness score (higher is better)
    """
    # Check valid selection
    if individual.n_selected() == 0:
        return -999.0
    
    # Select features
    feature_mask = individual.chromosome.astype(bool)
    X_selected = X.iloc[:, feature_mask]
    
    # Objective 1: Accuracy (maximize)
    model = Ridge(alpha=1.0, random_state=42)
    cv_scores = cross_val_score(
        model, X_selected, y, 
        cv=5, 
        scoring='neg_mean_squared_error'
    )
    # Convert to R² approximation (0-1 scale)
    mse = -cv_scores.mean()
    baseline_var = np.var(y)
    r2 = 1 - (mse / baseline_var)
    accuracy_score = max(0, r2)  # Ensure non-negative
    
    # Objective 2: Parsimony (minimize features)
    parsimony_penalty = individual.n_selected() / len(individual.chromosome)
    
    # Objective 3: Redundancy (minimize feature correlation)
    if individual.n_selected() > 1:
        corr_matrix = X_selected.corr().abs()
        # Get upper triangle (exclude diagonal)
        upper_triangle = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        mean_correlation = upper_triangle.stack().mean()
        redundancy_penalty = mean_correlation
    else:
        redundancy_penalty = 0.0
    
    # Store individual objectives
    individual.objectives = {
        'accuracy': accuracy_score,
        'parsimony': parsimony_penalty,
        'redundancy': redundancy_penalty
    }
    
    # Weighted sum (maximize accuracy, minimize others)
    fitness = (w_accuracy * accuracy_score - 
               w_parsimony * parsimony_penalty - 
               w_redundancy * redundancy_penalty)
    
    return fitness

# Test fitness function
data = load_diabetes()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X.columns
)

test_ind = Individual.random(X_train_scaled.shape[1])
while test_ind.n_selected() == 0:
    test_ind = Individual.random(X_train_scaled.shape[1])

fitness = weighted_sum_fitness(test_ind, X_train_scaled, y_train)
test_ind.fitness = fitness

print(f"Test individual: {test_ind}")
print(f"Objectives:")
for obj, value in test_ind.objectives.items():
    print(f"  {obj}: {value:.4f}")
print(f"Combined fitness: {fitness:.4f}")

### 1.2 Constraint-Based Fitness

Handle hard constraints using penalty functions.

In [ ]:
# Purpose: Implement constraint handling via penalties
# Key Concept: Heavy penalty for constraint violation

def constrained_fitness(individual: Individual,
                       X: pd.DataFrame,
                       y: np.ndarray,
                       max_features: int = 5,
                       min_features: int = 2,
                       max_correlation: float = 0.9) -> float:
    """
    Fitness with hard constraints enforced via penalties.
    
    Constraints:
    1. Must select between min_features and max_features
    2. No feature pair can have correlation > max_correlation
    
    Parameters
    ----------
    individual : Individual
        Candidate solution
    X : DataFrame
        Feature matrix
    y : array
        Target variable
    max_features : int
        Maximum allowed features (default 5)
    min_features : int
        Minimum required features (default 2)
    max_correlation : float
        Maximum allowed pairwise correlation (default 0.9)
    
    Returns
    -------
    fitness : float
        Fitness with penalties for violations
    """
    n_selected = individual.n_selected()
    
    # Constraint 1: Feature count bounds
    if n_selected < min_features or n_selected > max_features:
        # Heavy penalty
        violation = max(min_features - n_selected, n_selected - max_features, 0)
        return -100.0 * violation
    
    # Select features
    feature_mask = individual.chromosome.astype(bool)
    X_selected = X.iloc[:, feature_mask]
    
    # Constraint 2: Maximum correlation
    if n_selected > 1:
        corr_matrix = X_selected.corr().abs()
        upper_triangle = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        max_corr = upper_triangle.stack().max()
        
        if max_corr > max_correlation:
            # Penalty proportional to violation
            return -50.0 * (max_corr - max_correlation)
    
    # If all constraints satisfied, evaluate normally
    model = Ridge(alpha=1.0, random_state=42)
    cv_scores = cross_val_score(
        model, X_selected, y,
        cv=5,
        scoring='neg_mean_squared_error'
    )
    
    # Return negative MSE (higher is better)
    return cv_scores.mean()

# Test constrained fitness
test_ind = Individual(np.array([1, 1, 1, 0, 0, 0, 0, 0, 0, 0]))  # 3 features
fitness_constrained = constrained_fitness(
    test_ind, X_train_scaled, y_train,
    max_features=5, min_features=2
)

print(f"Test individual with {test_ind.n_selected()} features")
print(f"Constrained fitness: {fitness_constrained:.4f}")

# Test violation
test_ind_violate = Individual(np.array([1, 1, 1, 1, 1, 1, 1, 0, 0, 0]))  # 7 features (too many)
fitness_violate = constrained_fitness(
    test_ind_violate, X_train_scaled, y_train,
    max_features=5, min_features=2
)

print(f"\nViolating individual with {test_ind_violate.n_selected()} features")
print(f"Constrained fitness: {fitness_violate:.4f} (penalty applied)")

In [ ]:
section_divider("Time Series-Specific Fitness")

## 2. Time Series-Specific Fitness

### Key Concept: Time series forecasting requires special considerations.

1. **Temporal ordering**: Must respect time order in validation
2. **Walk-forward validation**: Use TimeSeriesSplit
3. **Multiple horizons**: Evaluate at different forecast horizons
4. **Stationarity**: Prefer features that improve stationarity

In [ ]:
callout("Time series forecasting requires special considerations.", kind="insight")

### 2.1 Time Series Cross-Validation Fitness

In [ ]:
# Purpose: Implement time series-aware fitness function
# Key Concept: Use TimeSeriesSplit to respect temporal order

def time_series_fitness(individual: Individual,
                       X: pd.DataFrame,
                       y: np.ndarray,
                       n_splits: int = 5,
                       parsimony_weight: float = 0.01) -> float:
    """
    Fitness for time series forecasting using walk-forward validation.
    
    Parameters
    ----------
    individual : Individual
        Candidate solution
    X : DataFrame
        Feature matrix (temporal order preserved)
    y : array
        Target variable (temporal order preserved)
    n_splits : int
        Number of time series splits (default 5)
    parsimony_weight : float
        Weight for parsimony penalty (default 0.01)
    
    Returns
    -------
    fitness : float
        Fitness score (higher is better)
    """
    if individual.n_selected() == 0:
        return -999.0
    
    # Select features
    feature_mask = individual.chromosome.astype(bool)
    X_selected = X.iloc[:, feature_mask]
    
    # Time series cross-validation
    tscv = TimeSeriesSplit(n_splits=n_splits)
    model = Ridge(alpha=1.0, random_state=42)
    
    cv_scores = cross_val_score(
        model, X_selected, y,
        cv=tscv,
        scoring='neg_mean_squared_error'
    )
    
    # Average performance across splits
    mse = -cv_scores.mean()
    
    # Convert to R²-like score
    baseline_var = np.var(y)
    r2_score = 1 - (mse / baseline_var)
    
    # Apply parsimony penalty
    parsimony_penalty = parsimony_weight * (individual.n_selected() / len(individual.chromosome))
    
    fitness = r2_score - parsimony_penalty
    
    return fitness

# Test time series fitness
test_ind = Individual.random(X_train_scaled.shape[1])
while test_ind.n_selected() == 0:
    test_ind = Individual.random(X_train_scaled.shape[1])

fitness_ts = time_series_fitness(test_ind, X_train_scaled, y_train)

print(f"Time series fitness: {fitness_ts:.4f}")
print(f"Features selected: {test_ind.n_selected()}/{len(test_ind.chromosome)}")

### 2.2 Multi-Horizon Fitness

Evaluate forecast performance at multiple horizons (1-step, 5-step, 10-step ahead).

In [ ]:
# Purpose: Evaluate fitness across multiple forecast horizons
# Key Concept: Good features should work well at different horizons

def multi_horizon_fitness(individual: Individual,
                         X: pd.DataFrame,
                         y: np.ndarray,
                         horizons: List[int] = [1, 5, 10],
                         horizon_weights: List[float] = None) -> float:
    """
    Fitness based on performance across multiple forecast horizons.
    
    Parameters
    ----------
    individual : Individual
        Candidate solution
    X : DataFrame
        Feature matrix
    y : array
        Target variable
    horizons : list of int
        Forecast horizons to evaluate (default [1, 5, 10])
    horizon_weights : list of float or None
        Weights for each horizon (default equal weights)
    
    Returns
    -------
    fitness : float
        Weighted average performance across horizons
    """
    if individual.n_selected() == 0:
        return -999.0
    
    # Default equal weights
    if horizon_weights is None:
        horizon_weights = [1.0 / len(horizons)] * len(horizons)
    
    # Select features
    feature_mask = individual.chromosome.astype(bool)
    X_selected = X.iloc[:, feature_mask]
    
    horizon_scores = []
    
    for h in horizons:
        # Create shifted target for h-step ahead forecast
        # Note: In practice, you'd use lagged features properly
        # This is simplified for demonstration
        if len(y) > h:
            X_h = X_selected.iloc[:-h] if h > 0 else X_selected
            y_h = y[h:] if h > 0 else y
            
            # Ensure same length
            min_len = min(len(X_h), len(y_h))
            X_h = X_h.iloc[:min_len]
            y_h = y_h[:min_len]
            
            if len(X_h) < 20:  # Not enough data
                continue
            
            # Evaluate with time series CV
            tscv = TimeSeriesSplit(n_splits=3)
            model = Ridge(alpha=1.0, random_state=42)
            
            try:
                scores = cross_val_score(
                    model, X_h, y_h,
                    cv=tscv,
                    scoring='neg_mean_squared_error'
                )
                horizon_scores.append(-scores.mean())
            except:
                horizon_scores.append(np.inf)
    
    if not horizon_scores:
        return -999.0
    
    # Weighted average of horizon errors (lower is better)
    # Convert to fitness (higher is better)
    weighted_mse = np.average(horizon_scores, weights=horizon_weights[:len(horizon_scores)])
    
    # Normalize to 0-1 scale (approximate)
    baseline_var = np.var(y)
    fitness = 1 - (weighted_mse / baseline_var)
    
    return fitness

# Test multi-horizon fitness
test_ind = Individual.random(X_train_scaled.shape[1])
while test_ind.n_selected() == 0:
    test_ind = Individual.random(X_train_scaled.shape[1])

fitness_mh = multi_horizon_fitness(
    test_ind, X_train_scaled, y_train,
    horizons=[1, 3, 5],
    horizon_weights=[0.5, 0.3, 0.2]  # Prioritize short-term
)

print(f"Multi-horizon fitness: {fitness_mh:.4f}")
print(f"Features selected: {test_ind.n_selected()}/{len(test_ind.chromosome)}")

In [ ]:
section_divider("Fitness Approximation for Speed")

## 3. Fitness Approximation for Speed

### Key Concept: Full fitness evaluation can be expensive. Use approximations for speed.

Strategies:
1. **Data sampling**: Use subset of data
2. **Fewer CV folds**: Use 3-fold instead of 5-fold
3. **Surrogate models**: Train fast models to predict full fitness
4. **Fitness caching**: Store and reuse previous evaluations

In [ ]:
callout("Full fitness evaluation can be expensive. Use approximations for speed.", kind="insight")

### 3.1 Adaptive Fitness Evaluation

Use cheap approximation for most individuals, expensive evaluation for promising ones.

In [ ]:
# Purpose: Implement two-stage fitness evaluation
# Key Concept: Quick screening, then detailed evaluation for best candidates

class AdaptiveFitnessEvaluator:
    """
    Two-stage fitness evaluator for computational efficiency.
    
    Stage 1: Fast approximation (small data sample, fewer folds)
    Stage 2: Full evaluation (complete data, more folds)
    """
    
    def __init__(self, X: pd.DataFrame, y: np.ndarray):
        self.X_full = X
        self.y_full = y
        
        # Create small sample for quick evaluation
        sample_size = min(100, len(X) // 2)
        sample_idx = np.random.choice(len(X), size=sample_size, replace=False)
        self.X_sample = X.iloc[sample_idx]
        self.y_sample = y[sample_idx]
        
        # Cache for full evaluations
        self.cache = {}
    
    def quick_fitness(self, individual: Individual) -> float:
        """
        Fast fitness approximation using data sample.
        """
        if individual.n_selected() == 0:
            return -999.0
        
        # Use chromosome as cache key
        cache_key = tuple(individual.chromosome)
        if cache_key in self.cache:
            return self.cache[cache_key]['quick']
        
        feature_mask = individual.chromosome.astype(bool)
        X_selected = self.X_sample.iloc[:, feature_mask]
        
        # Quick model with 3-fold CV
        model = Ridge(alpha=1.0, random_state=42)
        scores = cross_val_score(
            model, X_selected, self.y_sample,
            cv=3,
            scoring='neg_mean_squared_error'
        )
        
        fitness = scores.mean()
        
        # Cache result
        if cache_key not in self.cache:
            self.cache[cache_key] = {}
        self.cache[cache_key]['quick'] = fitness
        
        return fitness
    
    def full_fitness(self, individual: Individual) -> float:
        """
        Complete fitness evaluation using full dataset.
        """
        if individual.n_selected() == 0:
            return -999.0
        
        # Check cache
        cache_key = tuple(individual.chromosome)
        if cache_key in self.cache and 'full' in self.cache[cache_key]:
            return self.cache[cache_key]['full']
        
        feature_mask = individual.chromosome.astype(bool)
        X_selected = self.X_full.iloc[:, feature_mask]
        
        # Full model with 5-fold CV
        model = Ridge(alpha=1.0, random_state=42)
        scores = cross_val_score(
            model, X_selected, self.y_full,
            cv=5,
            scoring='neg_mean_squared_error'
        )
        
        fitness = scores.mean()
        
        # Cache result
        if cache_key not in self.cache:
            self.cache[cache_key] = {}
        self.cache[cache_key]['full'] = fitness
        
        return fitness
    
    def evaluate_population(self, population: List[Individual], 
                          top_fraction: float = 0.2) -> None:
        """
        Evaluate population adaptively:
        1. Quick evaluation for all
        2. Full evaluation for top individuals
        
        Parameters
        ----------
        population : list of Individual
            Population to evaluate
        top_fraction : float
            Fraction of population to fully evaluate (default 0.2)
        """
        # Stage 1: Quick evaluation for all
        for ind in population:
            ind.fitness = self.quick_fitness(ind)
        
        # Stage 2: Full evaluation for top individuals
        n_full_eval = max(1, int(len(population) * top_fraction))
        sorted_pop = sorted(population, key=lambda ind: ind.fitness, reverse=True)
        
        for ind in sorted_pop[:n_full_eval]:
            ind.fitness = self.full_fitness(ind)

# Test adaptive evaluator
evaluator = AdaptiveFitnessEvaluator(X_train_scaled, y_train)

# Create test population
test_pop = [Individual.random(X_train_scaled.shape[1]) for _ in range(20)]
for ind in test_pop:
    while ind.n_selected() == 0:
        ind = Individual.random(X_train_scaled.shape[1])

print("Testing adaptive fitness evaluation...")
print(f"Population size: {len(test_pop)}")

import time
start = time.time()
evaluator.evaluate_population(test_pop, top_fraction=0.3)
elapsed = time.time() - start

print(f"Evaluation time: {elapsed:.2f} seconds")
print(f"Cache size: {len(evaluator.cache)} entries")
print(f"\nTop 3 individuals:")
sorted_pop = sorted(test_pop, key=lambda ind: ind.fitness, reverse=True)
for i, ind in enumerate(sorted_pop[:3], 1):
    print(f"  {i}. Fitness: {ind.fitness:.4f}, Features: {ind.n_selected()}")

In [ ]:
section_divider("Exercises")

## 4. Exercises

### Exercise 4.1: Custom Domain Fitness

**Task**: Design a fitness function for a medical diagnosis problem where:
- Minimizing false negatives is 3x more important than false positives
- Each lab test (feature) has an associated cost
- Total cost must not exceed a budget

**Expected Output**: Fitness function that balances diagnostic accuracy, false negative rate, and cost.

In [ ]:
# YOUR CODE HERE
# ---------------

def medical_diagnosis_fitness(individual: Individual,
                             X: pd.DataFrame,
                             y: np.ndarray,
                             feature_costs: np.ndarray,
                             budget: float,
                             fn_weight: float = 3.0) -> float:
    """
    Custom fitness for medical diagnosis with cost constraints.
    
    TODO: Implement
    - Calculate total cost of selected features
    - Penalize if budget exceeded
    - Evaluate using precision/recall
    - Weight false negatives heavily
    """
    pass

### Exercise 4.2: Pareto Front Visualization

**Task**: Implement a function to identify and visualize the Pareto front for two objectives (accuracy vs number of features).

**Expected Output**: Scatter plot showing Pareto-optimal solutions.

In [ ]:
# YOUR CODE HERE
# ---------------

def find_pareto_front(population: List[Individual]) -> List[Individual]:
    """
    Find Pareto-optimal solutions.
    
    A solution is Pareto-optimal if no other solution dominates it
    (better in all objectives).
    
    TODO: Implement dominance check and filter
    """
    pass

# TODO: Create plot showing Pareto front

### Exercise 4.3: Fitness Landscape Analysis

**Task**: Analyze the fitness landscape by:
1. Sampling random individuals
2. Computing fitness vs number of features
3. Identifying ruggedness (how much fitness varies with small changes)

**Expected Output**: Visualization and metrics describing landscape difficulty.

In [ ]:
# YOUR CODE HERE
# ---------------

In [ ]:
section_divider("Summary")

## 5. Summary

### Key Takeaways

1. **Multi-Objective**: Real problems have conflicting objectives - use weighted sum or Pareto optimization
2. **Constraints**: Handle via penalties (soft) or repair operators (hard)
3. **Time Series**: Respect temporal order, use TimeSeriesSplit, consider multiple horizons
4. **Computational Efficiency**: Use approximation, caching, and adaptive evaluation
5. **Domain Knowledge**: Incorporate expertise through custom fitness components
6. **Overfitting Prevention**: Parsimony pressure and proper cross-validation are essential

### Fitness Design Checklist

- [ ] All objectives clearly defined and measured
- [ ] Objectives normalized to similar scales
- [ ] Weights reflect true importance
- [ ] Constraints properly enforced
- [ ] Cross-validation prevents overfitting
- [ ] Computational cost acceptable
- [ ] Edge cases handled (empty selection, etc.)

### What's Next

In the next notebook, we'll focus on **overfitting prevention**:
- Holdout validation strategies
- Regularization in fitness
- Early stopping criteria
- Ensemble approaches

In [ ]:
key_takeaways(["Real problems have conflicting objectives - use weighted sum or Pareto optimization", "Handle via penalties (soft) or repair operators (hard)", "Respect temporal order, use TimeSeriesSplit, consider multiple horizons", "Use approximation, caching, and adaptive evaluation", "Incorporate expertise through custom fitness components", "Parsimony pressure and proper cross-validation are essential"])